# PCA + Ridge (PCR) Volatility Backtest

CPU-only walk-forward backtest using PCA-compressed raw lag features
instead of HAR aggregates. PCA is re-fit every 240 steps; Ridge is
re-fit at the same cadence.

In [ ]:
# ── Setup: clone repo, install deps ──
import os

REPO_URL = "https://github.com/jamesdchen/harxhar.git"
REPO_DIR = "/content/harxhar"
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
!pip install -q numpy pandas scikit-learn pyarrow numba tqdm

In [ ]:
# ── Parameters ──
DATA_PATH = "all30min"
HORIZON = 1
TRAIN_WINDOW_DAYS = 500
N_COMPONENTS = 5

PERIODS_PER_DAY = 48
TRAIN_WINDOW = TRAIN_WINDOW_DAYS * PERIODS_PER_DAY
print(f"Train window: {TRAIN_WINDOW} periods ({TRAIN_WINDOW_DAYS} days)")
print(f"PCA components: {N_COMPONENTS}")
print(f"Horizon: {HORIZON}")

In [ ]:
# ── Load + transform ──
import numpy as np
import pandas as pd
from src.loading import load_raw_data
from src.transforms import robust_transform

df = load_raw_data(DATA_PATH)
print(f"Loaded: {df.shape}")
print(f"Date range: {df['t'].min()} → {df['t'].max()}")

adj_series, baseline = robust_transform(df, "RV", is_target=True)
df["adj_RV"] = adj_series
df["baseline"] = baseline

print(f"\nadj_RV stats:")
display(df["adj_RV"].describe())

In [ ]:
# ── Log-spaced raw lag features ──
from src.ml_pcr import resolve_pca_lags, generate_raw_lag_features

lags = resolve_pca_lags()
print(f"Number of lags: {len(lags)}")
print(f"Lags: {lags}")

# Compare with HAR lags
har_lags = [1, 5, 22]  # standard HAR
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 3))
ax.scatter(lags, [1] * len(lags), marker="|", s=200, label="PCR lags (log-spaced)")
ax.scatter(har_lags, [0.8] * len(har_lags), marker="|", s=200, color="red", label="HAR lags")
ax.set_xscale("log")
ax.set_xlabel("Lag (periods)")
ax.set_yticks([])
ax.legend()
ax.set_title("Lag distribution: PCR vs HAR")
plt.tight_layout()
plt.show()

# Generate features
df, feature_names = generate_raw_lag_features(df, target_col="adj_RV")
print(f"\nFeature columns ({len(feature_names)}): {feature_names[:5]} ... {feature_names[-3:]}")
print(f"DataFrame shape: {df.shape}")

In [ ]:
# ── PCA transform: fit on sample, explained variance, scree plot ──
from sklearn.decomposition import PCA

# Horizon shift + drop NaN
df["target"] = df["adj_RV"].shift(-HORIZON)
max_lag = lags[-1]
df_clean = df.iloc[max_lag:].dropna(subset=["target"] + feature_names).reset_index(drop=True)
print(f"Clean rows: {len(df_clean):,}")

# Fit PCA on first train_window rows for inspection
X_sample = df_clean[feature_names].values[:TRAIN_WINDOW].astype(np.float64)
# Simple z-score for visualisation only
X_sample_z = (X_sample - X_sample.mean(axis=0)) / (X_sample.std(axis=0) + 1e-12)

pca_full = PCA(svd_solver="randomized")
pca_full.fit(X_sample_z)

evr = pca_full.explained_variance_ratio_
cum_evr = np.cumsum(evr)

print(f"\nExplained variance ratio (first {N_COMPONENTS}): {evr[:N_COMPONENTS].round(4)}")
print(f"Cumulative at {N_COMPONENTS} components: {cum_evr[N_COMPONENTS - 1]:.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.bar(range(1, len(evr) + 1), evr)
ax1.set_xlabel("Component")
ax1.set_ylabel("Explained variance ratio")
ax1.set_title("Scree plot")
ax1.axvline(N_COMPONENTS, color="red", ls="--", label=f"n_components={N_COMPONENTS}")
ax1.legend()

ax2.plot(range(1, len(cum_evr) + 1), cum_evr, "o-")
ax2.axhline(0.95, color="grey", ls="--", alpha=0.5, label="95%")
ax2.axvline(N_COMPONENTS, color="red", ls="--", label=f"n_components={N_COMPONENTS}")
ax2.set_xlabel("Component")
ax2.set_ylabel("Cumulative explained variance")
ax2.set_title("Cumulative variance")
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── Ridge on PCA features: walk-forward with periodic PCA refit ──
from src.ml_pcr import run_pcr_backtest

X = df_clean[feature_names].values.astype(np.float64)
y = df_clean["target"].values.astype(np.float64)
dates = df_clean["t"].values
baselines = df_clean["baseline"].values

print(f"X shape: {X.shape}")
print(f"Test samples: {len(y) - TRAIN_WINDOW:,}")
print(f"PCA refit every 240 steps")

forecasts = run_pcr_backtest(
    X, y,
    train_window=TRAIN_WINDOW,
    n_components=N_COMPONENTS,
    refit_frequency=240,
)
print(f"\nForecasts: {len(forecasts):,}")

In [ ]:
# ── Quick eval ──
y_test = y[TRAIN_WINDOW:]
dates_test = dates[TRAIN_WINDOW:]
baselines_test = baselines[TRAIN_WINDOW:]

# Adjusted-scale metrics
errors = y_test - forecasts
mse = np.mean(errors ** 2)
mae = np.mean(np.abs(errors))
print(f"Adjusted-scale MSE: {mse:.6f}")
print(f"Adjusted-scale MAE: {mae:.6f}")

# Duan smearing
smear = np.mean((y_test - forecasts) ** 2)
pred_raw = (forecasts ** 2 + smear) * baselines_test
true_raw = (y_test ** 2) * baselines_test

# QLIKE
mask = (true_raw > 0) & (pred_raw > 0)
ratio = true_raw[mask] / pred_raw[mask]
qlike = np.mean(ratio - np.log(ratio) - 1.0)
print(f"QLIKE (raw scale): {qlike:.6f}")

# Save results
results = pd.DataFrame({
    "date": dates_test,
    "horizon": HORIZON,
    "true_adj": y_test,
    "pred_adj": forecasts,
    "true_raw": true_raw,
    "pred_raw": pred_raw,
})
out_path = f"results_pcr_h{HORIZON}.csv"
results.to_csv(out_path, index=False)
print(f"\nSaved {len(results)} rows to {out_path}")
display(results.head())

In [ ]:
%%writefile ../src/ml_pcr.py
"""PCA + Ridge (PCR) walk-forward backtest for volatility forecasting.

Standalone module — no imports from core/ or projects/.
Uses: numpy, pandas, argparse, os, tqdm, sklearn, numba.
"""

import argparse
import os

import numpy as np
import pandas as pd
from numba import njit
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from tqdm import tqdm

from src.loading import load_raw_data
from src.transforms import robust_transform

# ── Constants ──────────────────────────────────────────────────────────
PERIODS_PER_DAY = 48


# ── Log-spaced lags ───────────────────────────────────────────────────


def resolve_pca_lags(max_lag: int = 3125, num_points: int = 20) -> list[int]:
    """Generate log-spaced lag indices from 1 to *max_lag*."""
    raw = np.geomspace(1, max_lag, num=num_points)
    return sorted(set(int(round(v)) for v in raw))


def generate_raw_lag_features(
    df: pd.DataFrame,
    target_col: str = "adj_RV",
    max_lag: int = 3125,
) -> tuple[pd.DataFrame, list[str]]:
    """Create shifted-lag columns for each log-spaced lag."""
    lags = resolve_pca_lags(max_lag)
    features: dict[str, pd.Series] = {}
    feature_names: list[str] = []
    for lag in lags:
        name = f"{target_col}_lag_{lag}"
        features[name] = df[target_col].shift(lag)
        feature_names.append(name)
    feat_df = pd.DataFrame(features, index=df.index)
    return pd.concat([df, feat_df], axis=1), feature_names


# ── PCA transform wrapper ────────────────────────────────────────────


class PCATransform:
    """Thin wrapper around sklearn PCA for the backtest loop."""

    def __init__(self, n_components: int = 5):
        self.pca = PCA(n_components=n_components, svd_solver="randomized")

    def fit(self, X: np.ndarray, y: np.ndarray | None = None) -> "PCATransform":
        self.pca.fit(X)
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        return self.pca.transform(X)


# ── Numba-accelerated rolling robust scaler ──────────────────────────


@njit(cache=True)
def _update_sorted_matrix(
    sorted_mat: np.ndarray, x_old: np.ndarray, x_new: np.ndarray
) -> None:
    """Replace *x_old* with *x_new* in each feature's sorted window."""
    n_features, w = sorted_mat.shape
    for i in range(n_features):
        v_old = x_old[i]
        v_new = x_new[i]
        idx_old = np.searchsorted(sorted_mat[i], v_old)
        idx_new = np.searchsorted(sorted_mat[i], v_new)
        if idx_old < idx_new:
            idx_new -= 1
            for j in range(idx_old, idx_new):
                sorted_mat[i, j] = sorted_mat[i, j + 1]
        elif idx_old > idx_new:
            for j in range(idx_old, idx_new, -1):
                sorted_mat[i, j] = sorted_mat[i, j - 1]
        sorted_mat[i, idx_new] = v_new


@njit(cache=True)
def _get_robust_stats(sorted_mat: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Compute median and IQR from pre-sorted rolling window."""
    n_features, w = sorted_mat.shape
    median = np.empty(n_features, dtype=np.float64)
    iqr = np.empty(n_features, dtype=np.float64)
    idx_25 = (w - 1) * 0.25
    idx_50 = (w - 1) * 0.50
    idx_75 = (w - 1) * 0.75
    i25_floor, rem_25 = int(idx_25), idx_25 - int(idx_25)
    i50_floor, rem_50 = int(idx_50), idx_50 - int(idx_50)
    i75_floor, rem_75 = int(idx_75), idx_75 - int(idx_75)
    for i in range(n_features):
        q25 = sorted_mat[i, i25_floor] * (1.0 - rem_25) + sorted_mat[i, min(i25_floor + 1, w - 1)] * rem_25
        med = sorted_mat[i, i50_floor] * (1.0 - rem_50) + sorted_mat[i, min(i50_floor + 1, w - 1)] * rem_50
        q75 = sorted_mat[i, i75_floor] * (1.0 - rem_75) + sorted_mat[i, min(i75_floor + 1, w - 1)] * rem_75
        median[i] = med
        iq = q75 - q25
        iqr[i] = iq if iq >= 1e-12 else 1.0
    return median, iqr


class RollingRobustScaler:
    """Online robust scaler backed by sorted-matrix quantile tracking."""

    def __init__(self, window: int):
        self.window = window
        self.sorted_mat: np.ndarray | None = None
        self.buffer: np.ndarray | None = None
        self.pos: int = 0
        self.full: bool = False

    def initialize(self, X_init: np.ndarray) -> None:
        """Warm-start with an (n_samples, n_features) array."""
        n, p = X_init.shape
        self.window = n
        self.buffer = X_init.copy()
        self.sorted_mat = np.empty((p, n), dtype=np.float64)
        for j in range(p):
            self.sorted_mat[j] = np.sort(X_init[:, j])
        self.pos = 0
        self.full = True

    def update(self, x_new: np.ndarray) -> None:
        """Slide in a new observation, evicting the oldest."""
        assert self.buffer is not None and self.sorted_mat is not None
        x_old = self.buffer[self.pos].copy()
        self.buffer[self.pos] = x_new
        _update_sorted_matrix(self.sorted_mat, x_old, x_new)
        self.pos = (self.pos + 1) % self.window

    def transform_single(self, x: np.ndarray) -> np.ndarray:
        """Scale a single observation using current median/IQR."""
        assert self.sorted_mat is not None
        median, iqr = _get_robust_stats(self.sorted_mat)
        return (x - median) / iqr

    def transform_buffer(self) -> np.ndarray:
        """Scale the entire current buffer."""
        assert self.buffer is not None and self.sorted_mat is not None
        median, iqr = _get_robust_stats(self.sorted_mat)
        return (self.buffer - median) / iqr


# ── Walk-forward PCR backtest ─────────────────────────────────────────


def run_pcr_backtest(
    X: np.ndarray,
    y: np.ndarray,
    train_window: int,
    n_components: int = 5,
    refit_frequency: int = 240,
    alpha: float = 1.0,
) -> np.ndarray:
    """Walk-forward PCA + Ridge backtest.

    Parameters
    ----------
    X : (N, p) feature matrix (raw lag features, already winsorized/transformed).
    y : (N,) target vector.
    train_window : int
        Number of observations in the rolling training window.
    n_components : int
        PCA dimensionality.
    refit_frequency : int
        Re-fit PCA every this many steps (Ridge is also re-fit at PCA refit).
    alpha : float
        Ridge regularisation strength.

    Returns
    -------
    forecasts : (N - train_window,) array of predictions.
    """
    N, p = X.shape
    n_test = N - train_window
    forecasts = np.empty(n_test, dtype=np.float64)

    # ── Initialise scaler on first window ──
    scaler = RollingRobustScaler(window=train_window)
    scaler.initialize(X[:train_window])

    # ── Fit PCA on scaled buffer ──
    pca = PCATransform(n_components=n_components)
    X_buf_scaled = scaler.transform_buffer()
    pca.fit(X_buf_scaled)

    # ── Fit Ridge on PCA-transformed buffer ──
    X_buf_pca = pca.transform(X_buf_scaled)
    y_buf = y[:train_window]
    ridge = Ridge(alpha=alpha, fit_intercept=True)
    ridge.fit(X_buf_pca, y_buf)

    steps_since_refit = 0

    for i in tqdm(range(n_test), desc="PCR backtest", leave=False):
        idx = train_window + i
        x_t = X[idx]

        # Scale and PCA-transform the new observation
        x_scaled = scaler.transform_single(x_t)
        x_pca = pca.transform(x_scaled.reshape(1, -1))

        # Predict
        forecasts[i] = ridge.predict(x_pca)[0]

        # Update scaler buffer
        scaler.update(x_t)
        steps_since_refit += 1

        # Periodic PCA + Ridge refit
        if steps_since_refit >= refit_frequency:
            X_buf_scaled = scaler.transform_buffer()
            pca.fit(X_buf_scaled)
            X_buf_pca = pca.transform(X_buf_scaled)

            # Reconstruct y buffer in correct order
            buf_start = idx + 1 - train_window
            y_buf = y[buf_start : idx + 1]
            ridge.fit(X_buf_pca, y_buf)
            steps_since_refit = 0

    return forecasts


# ── CLI entry point ───────────────────────────────────────────────────


def main() -> None:
    parser = argparse.ArgumentParser(description="PCA + Ridge (PCR) volatility backtest")
    parser.add_argument("--data-path", type=str, required=True, help="Path to parquet data")
    parser.add_argument("--horizon", type=int, default=1, help="Forecast horizon in periods")
    parser.add_argument("--train-window", type=int, default=500, help="Training window in days")
    parser.add_argument("--chunk-id", type=int, default=0, help="Chunk index for parallel runs")
    parser.add_argument("--total-chunks", type=int, default=1, help="Total number of chunks")
    parser.add_argument("--output-file", type=str, default="results_pcr.csv", help="Output CSV path")
    parser.add_argument("--n-components", type=int, default=5, help="Number of PCA components")
    args = parser.parse_args()

    train_window = args.train_window * PERIODS_PER_DAY

    # ── Load data ─────────────────────────────────────────────────────
    df = load_raw_data(args.data_path)

    # ── Transform target ──────────────────────────────────────────────
    adj_series, baseline = robust_transform(df, "RV", is_target=True)
    df["adj_RV"] = adj_series
    df["baseline"] = baseline

    # ── Generate raw lag features ─────────────────────────────────────
    df, feature_names = generate_raw_lag_features(df, target_col="adj_RV")

    # ── Horizon shift ─────────────────────────────────────────────────
    df["target"] = df["adj_RV"].shift(-args.horizon)

    # ── Drop NaN rows (from lags + horizon shift) ─────────────────────
    max_lag = resolve_pca_lags()[-1]
    df = df.iloc[max_lag:].reset_index(drop=True)
    df = df.dropna(subset=["target"] + feature_names).reset_index(drop=True)

    # ── Chunk selection ───────────────────────────────────────────────
    n_total = len(df) - train_window
    chunk_size = n_total // args.total_chunks
    start = args.chunk_id * chunk_size
    end = n_total if args.chunk_id == args.total_chunks - 1 else (args.chunk_id + 1) * chunk_size

    # Slice so that train window precedes the chunk's test range
    df_chunk = df.iloc[start : train_window + end].reset_index(drop=True)

    # ── Prepare arrays ────────────────────────────────────────────────
    X = df_chunk[feature_names].values.astype(np.float64)
    y = df_chunk["target"].values.astype(np.float64)
    dates = df_chunk["t"].values
    baselines = df_chunk["baseline"].values

    # ── Run backtest ──────────────────────────────────────────────────
    forecasts = run_pcr_backtest(
        X, y,
        train_window=train_window,
        n_components=args.n_components,
        refit_frequency=240,
    )

    # ── Duan smearing + save ──────────────────────────────────────────
    y_test = y[train_window:]
    dates_test = dates[train_window:]
    baselines_test = baselines[train_window:]

    smear = np.mean((y_test - forecasts) ** 2)
    pred_raw = (forecasts ** 2 + smear) * baselines_test
    true_raw = (y_test ** 2) * baselines_test

    results = pd.DataFrame({
        "date": dates_test,
        "horizon": args.horizon,
        "true_adj": y_test,
        "pred_adj": forecasts,
        "true_raw": true_raw,
        "pred_raw": pred_raw,
    })

    os.makedirs(os.path.dirname(args.output_file) or ".", exist_ok=True)
    results.to_csv(args.output_file, index=False)
    print(f"Saved {len(results)} rows to {args.output_file}")


if __name__ == "__main__":
    main()